# DM-NeRF / DM-SR — study (Kaggle, Resume)
Ative GPU e Internet na interface e execute todas as células. Não edite código.
Na continuação, anexe em Add Input → Notebook Output a saída anterior deste projeto.
Use Save Version → Save & Run All para preservar saídas.
O início executa smoke test e piloto de até uma hora. A continuação retoma o treino até o orçamento da sessão.
Uma sessão gratuita não garante convergência. Somente study é executada; nenhuma decisão congelada é alterada.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

def run(*args):
    subprocess.run([str(x) for x in args], check=True)

run('nvidia-smi')
BASE=Path('/kaggle/temp/dissertacao-dmnerf')
OUTPUT=Path('/kaggle/working/dmnerf-results')
if not (OUTPUT/'raw/study/experiment/latest.tar').exists():
    candidates=[]
    for state in Path('/kaggle/input').rglob('training_state.json'):
        if state.parent.name=='experiment' and (state.parent/'identity.json').is_file() and (state.parent/'latest.tar').is_file():
            candidates.append((json.loads(state.read_text())['iteration'],state.parents[3]))
    if not candidates:
        raise RuntimeError('Anexe a saída do piloto/continuação em Add Input → Notebook Output.')
    previous=sorted(candidates,key=lambda item:(item[0],str(item[1])))[-1][1]
    shutil.copytree(previous,OUTPUT,dirs_exist_ok=True)
if not (OUTPUT/'raw/study/experiment/latest.tar').exists():
    raise RuntimeError('Execute primeiro o notebook Study: checkpoint piloto ausente.')
BASE.mkdir(parents=True,exist_ok=True)
REPO=BASE/'sdf-generator'
WORK=BASE/'work'
if not REPO.exists():
    run('git','clone','--branch','dissertacao-2026','--single-branch',
        'https://github.com/Jfsslemos/sdf-generator.git',REPO)
run('git','-C',REPO,'rev-parse','HEAD')
lock=['--lock',OUTPUT/'environment.freeze.txt'] if (OUTPUT/'environment.freeze.txt').exists() else []
run(sys.executable,REPO/'tools/dmnerf/bootstrap.py','--work',WORK,*lock)
PYTHON=WORK/'env/bin/python'
run(PYTHON,'-c','import torch; assert torch.cuda.is_available(), "Ative GPU"; print(torch.cuda.get_device_name(0))')


In [ ]:
run(PYTHON,REPO/'tools/dmnerf/prepare.py','--work',WORK)


In [ ]:
run(PYTHON,REPO/'tools/dmnerf/run.py','--work',WORK,'--output',OUTPUT,'--profile','full')


In [ ]:
print('Resultados:',OUTPUT)
for state in sorted(OUTPUT.rglob('training_state.json')):
    print(state.relative_to(OUTPUT),json.loads(state.read_text()))
print('Pacote:',OUTPUT.parent/(OUTPUT.name+'-bundle.zip'))


## Continuação
Envie o pacote de resultados para revisar tempo/VRAM e decidir viabilidade. No Kaggle, preserve a versão e anexe sua saída ao notebook Resume. No Colab, use Resume com o mesmo Drive.
O smoke é diagnóstico. Avaliação completa e meshing 256³ só ocorrem ao completar o treino. Ausência de superfície é registrada, não corrigida por mudança arbitrária do limiar.
Veja `docs/GPU_RUNBOOK.md`.